In [1]:
import pandas as pd
from pathlib import Path

In [2]:
base_path = Path().resolve().parent
file_path = base_path / 'data' / 'raw'

In [3]:
# Student observations
obs = pd.read_excel(
    file_path / 'Stellar_edu_MDS_ap_stats_dataset - v1.9.xlsx', 
    sheet_name='Student_Observations')


# Fine KC to Modeling KC mapping
kc_map = pd.read_excel(
    file_path / "mkc_mapping_pack_v1.0..xlsx",
    sheet_name="FineKC_to_ModelingKC_Map"
)


# Modeling KC weights
weights = pd.read_excel(
    file_path / "mkc_weights_dataset.xlsx",
    sheet_name="MKC_Weights"
)

# BKT predictions from Mailys
bkt = pd.read_csv(
    file_path / "mkc_data-june01.csv"   
)


In [4]:
print("=== SHAPES ===")
print(f"Observations : {obs.shape}")
print(f"KC Map       : {kc_map.shape}")
print(f"Weights      : {weights.shape}")
print(f"BKT preds    : {bkt.shape}")


=== SHAPES ===
Observations : (21808, 14)
KC Map       : (256, 12)
Weights      : (47, 18)
BKT preds    : (1963, 29)


In [5]:
# STEP 1:  ADD MODELING KC VIA MAPPING
#  Joining: obs.primary_kc_id to kc_map.fine_kc_id

In [6]:
kc_map_slim = kc_map[[
    "fine_kc_id",
    "fine_kc_label",
    "modeling_kc_id",
    "modeling_kc_label",
    "modeling_unit",
    "fine_reporting_group"
]].drop_duplicates()

In [7]:
kc_map_slim.shape

(256, 6)

In [8]:
df = obs.merge(
    kc_map_slim,
    left_on="primary_kc_id",
    right_on="fine_kc_id",
    how="left"         
)

In [9]:
unmatched = df["modeling_kc_id"].isna().sum()
print(f"Step 1 has; {df.shape[0]} rows | Unmatched fine KCs: {unmatched}")

Step 1 has; 21808 rows | Unmatched fine KCs: 0


In [10]:
df.columns

Index(['student_id', 'assignment_id', 'class_num', 'observation_id',
       'item_type', 'source_question', 'primary_kc_id', 'all_kc_ids', 'score',
       'max_score', 'percent_score', 'student_response',
       'correct_answer_or_rubric', 'rubric_level', 'fine_kc_id',
       'fine_kc_label', 'modeling_kc_id', 'modeling_kc_label', 'modeling_unit',
       'fine_reporting_group'],
      dtype='str')

In [11]:
# STEP 2: ADD WEIGHTS
# Join on: modeling_kc_id

df_2 = df.merge(
    weights,
    on="modeling_kc_id",
    how="left"
)

In [12]:
df_2.head(2)

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,...,estimated_exam_share_pct,unit_exam_weighting,direct_dependents,downstream_dependents,weighted_direct_edge_count,num_primary_items,num_student_records,mean_percent_score,weight_rationale,2026_27_revision_note
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.observational_unit_variable,KC.U1.02.observational_unit_variable,0.0,1,...,2.42,15%–23%,4,41,4,11,346,0.6257,foundation for 41 downstream MKCs; high AP uni...,NaN
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,...,2.42,15%–23%,4,41,4,11,346,0.6257,foundation for 41 downstream MKCs; high AP uni...,NaN


In [13]:
unmatched_w = df_2["weight"].isna().sum()
print(f"Data shape: {df_2.shape[0]} rows | Rows missing weight: {unmatched_w}")


Data shape: 21808 rows | Rows missing weight: 0


In [14]:
#  ADD BKT PREDICTIONS
#  Join on: student_id = user_id
#  modeling_kc_id = skill_name

bkt_preds = bkt[[
        "user_id",
        "skill_name",
        "correct",
        "order_id",
        "correct_predictions",
        "state_predictions",
    ]]

# Renaming BKT columns to align with our dataframe
bkt_renamed = bkt_preds.rename(columns={
    "user_id"             : "student_id",
    "skill_name"          : "modeling_kc_id"
})



In [15]:
dupes = bkt_renamed.duplicated(subset=["student_id", "modeling_kc_id"]).sum()
print(f"Duplicate student + KC rows in BKT: {dupes}")

Duplicate student + KC rows in BKT: 1085


In [16]:
# checking order_id ranking and duplicates
sample = bkt_renamed[
    bkt_renamed.duplicated(subset=["student_id", "modeling_kc_id"], keep=False)
].sort_values(["student_id", "modeling_kc_id", "order_id"])

print(sample.head(20).to_string())

   student_id                                  modeling_kc_id  correct  order_id  correct_predictions  state_predictions
15       S001           MKC.U1.04.summary_statistics_outliers        1        15             0.534654           0.525598
32       S001           MKC.U1.04.summary_statistics_outliers        1        32             0.589230           0.690359
50       S001           MKC.U1.04.summary_statistics_outliers        1        50             0.631053           0.816618
71       S001           MKC.U1.04.summary_statistics_outliers        0        71             0.658206           0.898591
24       S001       MKC.U1.05.standardization_transformations        1        24             0.502956           0.334601
30       S001       MKC.U1.05.standardization_transformations        1        30             0.521083           0.428409
7        S001      MKC.U1.06.normal_distribution_calculations        1         7             0.558269           0.673843
14       S001      MKC.U1.06.nor

In [17]:
""" 
There are duplicate student ids and Modelling KCs
Kindly remember that order_id tells us the sequence, so sorting descending and taking the first row 
gives us the most recent/final mastery estimate per student per KC, that's why I filtered for the most recent.
"""

bkt_latest = bkt_renamed.sort_values("order_id", ascending=False).drop_duplicates(subset=["student_id", "modeling_kc_id"])


In [18]:
bkt_latest.shape

(878, 6)

In [19]:
dupes2 = bkt_latest.duplicated(subset=["student_id", "modeling_kc_id"]).sum()
print(f"Duplicate student + KC rows in BKT: {dupes2}")

Duplicate student + KC rows in BKT: 0


In [20]:
# Clean both keys before merging
# df_2["modeling_kc_id"] = df_2["modeling_kc_id"].str.strip().str.lower()
# bkt_latest["modeling_kc_id"] = bkt_latest["modeling_kc_id"].str.strip().str.lower()
# df_2["student_id"] = df_2["student_id"].astype(str).str.strip()
# bkt_latest["student_id"] = bkt_latest["student_id"].astype(str).str.strip()

In [21]:
df_final = df_2.merge(
    bkt_latest,
    on=["student_id", "modeling_kc_id"],
    how="left"
)


In [22]:
df_final.head(2)

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,...,weighted_direct_edge_count,num_primary_items,num_student_records,mean_percent_score,weight_rationale,2026_27_revision_note,correct,order_id,correct_predictions,state_predictions
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.observational_unit_variable,KC.U1.02.observational_unit_variable,0.0,1,...,4,11,346,0.6257,foundation for 41 downstream MKCs; high AP uni...,NaN,0.0,69.0,0.57258,0.502561
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,...,4,11,346,0.6257,foundation for 41 downstream MKCs; high AP uni...,NaN,0.0,69.0,0.57258,0.502561


In [23]:
unmatched_bkt = df_final["state_predictions"].isna().sum()
print(f" Data Shape:{df_final.shape[0]} rows | Rows missing BKT predictions: {unmatched_bkt}")

 Data Shape:21808 rows | Rows missing BKT predictions: 3485


In [24]:
df_final.to_csv(base_path / "data" / "processed" / "final_student_kc_data.csv", index=False)

In [25]:
df_final.head(2)

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,...,weighted_direct_edge_count,num_primary_items,num_student_records,mean_percent_score,weight_rationale,2026_27_revision_note,correct,order_id,correct_predictions,state_predictions
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.observational_unit_variable,KC.U1.02.observational_unit_variable,0.0,1,...,4,11,346,0.6257,foundation for 41 downstream MKCs; high AP uni...,NaN,0.0,69.0,0.57258,0.502561
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,...,4,11,346,0.6257,foundation for 41 downstream MKCs; high AP uni...,NaN,0.0,69.0,0.57258,0.502561


### Checking why we have some missing BKT predictions after the merge

In [26]:
# DIAGNOSTIC Checking why we have some missing BKT predictions after the merge

# What modeling_kc_ids exist in our merged df?
df_kcs = set(df_final["modeling_kc_id"].dropna().unique())

# What modeling_kc_ids exist in the BKT data?
bkt_kcs = set(bkt_latest["modeling_kc_id"].dropna().unique())

print("KCs in df but NOT in BKT:", df_kcs - bkt_kcs)
print("KCs in BKT but NOT in df:", bkt_kcs - df_kcs)
print(f"\nTotal KCs in df  : {len(df_kcs)}")
print(f"Total KCs in BKT : {len(bkt_kcs)}")


KCs in df but NOT in BKT: set()
KCs in BKT but NOT in df: set()

Total KCs in df  : 47
Total KCs in BKT : 47


In [27]:
# Check student IDs match
df_students  = set(df_final["student_id"].dropna().unique())
bkt_students = set(bkt_latest["student_id"].dropna().unique())

print("\nStudents in df but NOT in BKT:", df_students - bkt_students)
print("Students in BKT but NOT in df:", bkt_students - df_students)


Students in df but NOT in BKT: set()
Students in BKT but NOT in df: set()


In [28]:
len(bkt_latest["student_id"].unique())

25

In [29]:
len(bkt_renamed["modeling_kc_id"].unique())

47

In [30]:
# Get all unique students and KCs from BKT data
all_students = bkt_latest["student_id"].unique()
all_kcs      = bkt_latest["modeling_kc_id"].unique()

# Build the full cartesian product (every possible combo)
full_grid = pd.MultiIndex.from_product(
    [all_students, all_kcs],
    names=["student_id", "modeling_kc_id"]
).to_frame(index=False)

print(f"Full grid (25 students × 47 KCs) : {full_grid.shape[0]} rows")

# Find which combos exist in BKT
bkt_combos = bkt_latest[["student_id", "modeling_kc_id"]].drop_duplicates()

# Anti-join combos in full grid but NOT in BKT
missing_combos = full_grid.merge(
    bkt_combos,
    on=["student_id", "modeling_kc_id"],
    how="left",
    indicator=True
).query('_merge == "left_only"').drop(columns="_merge")

print(f"Missing combos : {missing_combos.shape[0]}")

Full grid (25 students × 47 KCs) : 1175 rows
Missing combos : 297


Summary;
175 possible combinations (25 × 47)
878 combos exist in BKT (students who actually attempted that KC)
297 combos missing (students who simply never had a question on that KC)

In [31]:
# Checking if the KC names actually match between df and BKT
df_kcs  = set(df_2["modeling_kc_id"].dropna().unique())
bkt_kcs = set(bkt_latest["modeling_kc_id"].dropna().unique())

print("KCs in df but NOT in BKT:", df_kcs - bkt_kcs)
print("KCs in BKT but NOT in df:", bkt_kcs - df_kcs)

KCs in df but NOT in BKT: set()
KCs in BKT but NOT in df: set()
